# Tree-Ring Watermarks: a complete teaching example in Google Colab

**Generation → invisible watermark → detection → attacks → detection again.**

Open this `.ipynb` using **Colab → File → Upload notebook**, select
**Runtime → Change runtime type → GPU**, then **Runtime → Run all**.
The target configuration is a cloud GPU with 12–16 GB VRAM and about 12 GB system RAM.
This is a target budget, not a measured peak: the notebook reports actual memory usage as it runs.
On a Mac, you only need a browser and an internet connection. The first run downloads several GB of weights into Colab.

Defaults: 512×512, FP16, one image at a time, 50 steps, CPU offload;
no training, quantization, datasets, W&B, or additional evaluation models.
Execution may take tens of minutes. After the first detection passes, the notebook estimates
the time needed for the remaining attacks. Results are saved incrementally; download the ZIP before the session ends.

**Based on:** [Wen et al., Tree-Ring Watermarks (2023)](https://arxiv.org/abs/2305.20030),
[official implementation](https://github.com/YuxinWenRick/tree-ring-watermark).
This is a transparent adaptation for modern libraries, **not a full reproduction of the paper's tables**.
Attack success and watermark survival are computed at runtime, not predetermined.

## 1. Installation

Start with a fresh GPU runtime. Keep the CUDA-enabled PyTorch supplied by Colab.
If you have already imported a different version of `diffusers` or `transformers`, restart
the runtime after installation and continue with section 2.

In [ ]:
import subprocess
import sys
from importlib import metadata

PINNED = {
    "diffusers": "0.35.1",
    "transformers": "4.51.3",
    "accelerate": "1.10.1",
    "huggingface-hub": "0.34.4",
    "safetensors": "0.6.2",
}
already_imported = {
    name: sys.modules[name.replace("-", "_")].__version__
    for name in PINNED
    if name.replace("-", "_") in sys.modules
    and hasattr(sys.modules[name.replace("-", "_")], "__version__")
}
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    *[f"{name}=={version}" for name, version in PINNED.items()],
    "matplotlib>=3.7,<4", "pandas>=2,<3", "scipy>=1.11,<2",
    "Pillow>=10,<12", "psutil>=5.9,<8", "tqdm>=4.66,<5",
])
stale = [name for name, version in already_imported.items() if version != PINNED[name]]
if stale:
    raise RuntimeError(f"Restart the runtime after installation: {stale}. Then continue with section 2.")
print("Libraries installed. PyTorch was not reinstalled.")

## 2. Configuration and memory check

In [ ]:
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"  # public mirror; no token required
os.environ["USE_TF"] = "0"  # this notebook uses PyTorch without importing TensorFlow/Keras
os.environ["USE_FLAX"] = "0"
import gc
import io
import json
import math
import platform
import shutil
import time
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import torch
from PIL import Image, ImageFilter, ImageEnhance
from scipy.stats import ncx2, rankdata
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from diffusers import StableDiffusionPipeline, DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor2_0
from huggingface_hub import snapshot_download

PINNED = {
    "diffusers": "0.35.1", "transformers": "4.51.3", "accelerate": "1.10.1",
    "huggingface-hub": "0.34.4", "safetensors": "0.6.2",
}
for package, version in PINNED.items():
    if metadata.version(package) != version:
        raise RuntimeError(f"{package} version differs from {version}: run the installation in section 1.")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# @title Experiment settings
MODEL_ID = "sd2-community/stable-diffusion-2-1-base"
MODEL_REVISION = "4e63672c03103b6c636b8fb4119ba982469b2955"  # fixed mirror revision
PROMPT = "A photo of a red fox sitting in a snowy forest, natural light, detailed wildlife photography"
IMAGE_SIZE = 512
GENERATION_STEPS = 50
DETECTION_STEPS = 50
GUIDANCE_SCALE = 7.5
IMAGE_SEED = 2026
KEY_SEED = 999
W_RADIUS = 10
W_CHANNEL = 3
ALPHA = 0.01  # model-based p-value threshold from section 3.4 of the paper
MEMORY_MODE = "offload"  # "offload" saves VRAM; "gpu" is faster when memory allows
RUN_STRENGTH_SWEEP = False  # extra JPEG / crop / rotation settings; requires more inversions
RUN_MINI_BENCHMARK = False  # additional image pairs and ROC/AUC; slower
BENCHMARK_PAIRS = 6
DOWNLOAD_ZIP = False  # True: Colab offers to download the ZIP after execution

RUN_ID = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
OUTPUT_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
RESULTS = OUTPUT_ROOT / f'tree_ring_results_{RUN_ID}'
RESULTS.mkdir(parents=True, exist_ok=True)
assert IMAGE_SIZE == 512, "Keep 512 for this example: the radius and settings are chosen for SD 2.1-base."
assert 2 <= GENERATION_STEPS <= 100 and 2 <= DETECTION_STEPS <= 100
assert MEMORY_MODE in {"offload", "gpu"}
assert 0 < ALPHA < 1 and BENCHMARK_PAIRS >= 2
print("Results:", RESULTS)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. In Colab select Runtime → Change runtime type → GPU, then rerun.")
DEVICE = torch.device('cuda')
MODEL_DTYPE = torch.float16
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB")
print(f"System RAM: {psutil.virtual_memory().total / 2**30:.1f} GiB")
print(f"PyTorch {torch.__version__}; CUDA {torch.version.cuda}; Python {platform.python_version()}")
if gpu.total_memory < 10 * 2**30:
    print("Memory is below the target 12–16 GB: keep MEMORY_MODE='offload'.")

def memory_report(label):
    row = {"stage": label, "process_ram_gib": psutil.Process().memory_info().rss / 2**30}
    if torch.cuda.is_available():
        row.update(
            gpu_allocated_gib=torch.cuda.memory_allocated() / 2**30,
            gpu_reserved_gib=torch.cuda.memory_reserved() / 2**30,
            gpu_peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
            gpu_peak_reserved_gib=torch.cuda.max_memory_reserved() / 2**30,
        )
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in row.items()})
    return row

memory_rows = []
torch.cuda.reset_peak_memory_stats()
memory_rows.append(memory_report('before_loading'))

## 3. Load SD 2.1-base

Use a [public SD 2.1-base mirror](https://huggingface.co/sd2-community/stable-diffusion-2-1-base)
at a fixed commit. Download **only the required components as FP16 safetensors**,
excluding FP32 duplicates and large single-file checkpoints. Loading components incrementally reduces peak RAM.
The model was available without a token when this notebook was prepared; follow its Open RAIL++-M license.

Compute CFG with two sequential UNet calls instead of doubling the batch size.
SDPA and offload also work when calling components directly. Release offload hooks after each operation.
[Diffusers memory documentation](https://huggingface.co/docs/diffusers/v0.35.1/optimization/memory).

In [ ]:
model_dir = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=[
        'model_index.json', 'scheduler/scheduler_config.json', 'tokenizer/*',
        'text_encoder/config.json', 'text_encoder/model.fp16.safetensors',
        'unet/config.json', 'unet/diffusion_pytorch_model.fp16.safetensors',
        'vae/config.json', 'vae/diffusion_pytorch_model.fp16.safetensors',
    ],
    max_workers=2,
)
pipe = StableDiffusionPipeline.from_pretrained(
    model_dir, torch_dtype=MODEL_DTYPE, variant='fp16', use_safetensors=True,
    low_cpu_mem_usage=True, safety_checker=None, feature_extractor=None,
    requires_safety_checker=False, local_files_only=True,
)
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config, clip_sample=False, set_alpha_to_one=False,
    steps_offset=1, timestep_spacing='leading',
)
assert pipe.scheduler.config.prediction_type == 'epsilon', "Expected SD 2.1-base with epsilon prediction."
pipe.unet.set_attn_processor(AttnProcessor2_0())
pipe.enable_vae_slicing()
if MEMORY_MODE == 'offload':
    pipe.enable_model_cpu_offload()
else:
    pipe.to(DEVICE)
for module in (pipe.unet, pipe.vae, pipe.text_encoder):
    module.eval().requires_grad_(False)
LATENT_SIZE = IMAGE_SIZE // pipe.vae_scale_factor
LATENT_SHAPE = (1, pipe.unet.config.in_channels, LATENT_SIZE, LATENT_SIZE)
assert LATENT_SHAPE == (1, 4, 64, 64)
gc.collect()
memory_rows.append(memory_report('model_loaded'))
print('Latent shape:', LATENT_SHAPE, '| sampler: DDIM, eta=0')

## 4. Construct and embed Tree-Ring

Replace Fourier coefficients in a low-frequency circular region of one initial-noise channel.
The key consists of rings with constant complex values.
The mask, ring construction, and `.real` projection after IFFT follow the conventions in the official implementation:
[optim_utils.py](https://github.com/YuxinWenRick/tree-ring-watermark/blob/main/optim_utils.py).
FFT runs in FP32 on the CPU; this is a small array, not the full neural network.

The CPU-generated key is reproducible from its seed, but need not be bitwise identical to
a key generated by the original CUDA RNG. The detector owner knows the key.
The **noise** spectrum shown below is different from the spectrum of the final image.

In [ ]:
def fft2c(x):
    return torch.fft.fftshift(torch.fft.fft2(x.detach().cpu().float()), dim=(-2, -1))

def circle_mask(size, radius):
    # Use the coordinate convention from the authors, including the flipped y axis.
    y, x = np.ogrid[:size, :size]
    y = y[::-1]
    return torch.from_numpy(((x - size // 2)**2 + (y - size // 2)**2) <= radius**2)

def make_key(shape, seed, radius, channel):
    assert len(shape) == 4 and shape[0] == 1 and shape[-1] == shape[-2]
    assert 0 <= channel < shape[1] and 0 < radius < shape[-1] // 2
    g = torch.Generator(device='cpu').manual_seed(seed)
    source = fft2c(torch.randn(shape, generator=g))
    key = source.clone()
    for r in range(radius, 0, -1):
        disk = circle_mask(shape[-1], r)
        for c in range(shape[1]):
            key[:, c, disk] = source[0, c, 0, r].item()
    mask = torch.zeros(shape, dtype=torch.bool)
    mask[:, channel] = circle_mask(shape[-1], radius)
    return key, mask

def inject_watermark(noise, key, mask):
    spectrum = fft2c(noise)
    spectrum[mask] = key[mask]
    # As in the official implementation, project the complex key into real-valued noise.
    return torch.fft.ifft2(torch.fft.ifftshift(spectrum, dim=(-2, -1))).real.float()

def random_noise(seed):
    return torch.randn(LATENT_SHAPE, generator=torch.Generator(device='cpu').manual_seed(seed))

key, mask = make_key(LATENT_SHAPE, KEY_SEED, W_RADIUS, W_CHANNEL)
noise_plain = random_noise(IMAGE_SEED)
noise_marked = inject_watermark(noise_plain, key, mask)
print('Number of watermarked FFT coefficients:', int(mask.sum()))
print('Unmarked noise mean / std:', noise_plain.mean().item(), noise_plain.std().item())
print('Watermarked noise mean / std:', noise_marked.mean().item(), noise_marked.std().item())

In [ ]:
def plot_spectra(tensors, titles, filename):
    views = [torch.log1p(fft2c(x)[0, W_CHANNEL].abs()).numpy() for x in tensors]
    low, high = min(x.min() for x in views), max(x.max() for x in views)
    fig, axes = plt.subplots(1, len(views), figsize=(4.5 * len(views), 4), constrained_layout=True)
    for ax, view, title in zip(np.atleast_1d(axes), views, titles):
        im = ax.imshow(view, cmap='magma', vmin=low, vmax=high)
        ax.set_title(title)
        ax.axis('off')
    fig.colorbar(im, ax=np.atleast_1d(axes).tolist(), label='log(1 + |FFT|)', shrink=0.75)
    fig.savefig(RESULTS / filename, dpi=150)
    plt.show()
    plt.close(fig)

plot_spectra([noise_plain, noise_marked], ['Initial noise: unmarked', 'Initial noise: Tree-Ring'], '01_initial_spectra.png')

## 5. Generation and DDIM inversion

Deterministic DDIM transfers the state between noise levels:

\[
\hat x_0=(x_t-\sqrt{1-\alpha_t}\,\epsilon_\theta)/\sqrt{\alpha_t},\qquad
x_s=\sqrt{\alpha_s}\,\hat x_0+\sqrt{1-\alpha_s}\,\epsilon_\theta.
\]

Inversion traverses the levels in the opposite order, following
[inverse_stable_diffusion.py](https://github.com/YuxinWenRick/tree-ring-watermark/blob/main/inverse_stable_diffusion.py).
The detector receives **only the image and the key**: it does not use the original seed, noise, or prompt.
Inversion uses an empty prompt and guidance=1; image generation uses guidance=7.5.
UNet/VAE run in FP16, while the DDIM state and FFT use FP32 for numerical stability.

In [ ]:
@torch.inference_mode()
def text_embedding(prompt):
    tokens = pipe.tokenizer(
        prompt, padding='max_length', max_length=pipe.tokenizer.model_max_length,
        truncation=True, return_tensors='pt',
    )
    kwargs = {}
    if getattr(pipe.text_encoder.config, 'use_attention_mask', False):
        kwargs['attention_mask'] = tokens.attention_mask.to(DEVICE)
    return pipe.text_encoder(tokens.input_ids.to(DEVICE), **kwargs)[0].to(MODEL_DTYPE)

@torch.inference_mode()
def ddim_transfer(x, prediction, alpha_from, alpha_to):
    x0 = (x.float() - (1 - alpha_from).sqrt() * prediction.float()) / alpha_from.sqrt()
    return alpha_to.sqrt() * x0 + (1 - alpha_to).sqrt() * prediction.float()

@torch.inference_mode()
def diffusion_trajectory(initial, steps, *, inverse=False, prompt='', guidance=1.0):
    assert pipe.scheduler.config.prediction_type == 'epsilon'
    pipe.scheduler.set_timesteps(steps, device=DEVICE)
    timesteps = pipe.scheduler.timesteps.flip(0) if inverse else pipe.scheduler.timesteps
    alphas = pipe.scheduler.alphas_cumprod.to(device=DEVICE, dtype=torch.float32)
    alpha_final = pipe.scheduler.final_alpha_cumprod.to(device=DEVICE, dtype=torch.float32)
    step_ratio = pipe.scheduler.config.num_train_timesteps // steps
    unconditional = text_embedding('')
    conditional = text_embedding(prompt) if guidance > 1 else unconditional
    x = initial.to(device=DEVICE, dtype=torch.float32).clone()
    label = 'DDIM inversion' if inverse else 'DDIM generation'
    for t in tqdm(timesteps, desc=label, leave=False):
        ti = int(t.item())
        previous = ti - step_ratio
        alpha_from = alphas[ti]
        alpha_to = alphas[previous] if previous >= 0 else alpha_final
        if inverse:
            alpha_from, alpha_to = alpha_to, alpha_from
        model_input = x.to(MODEL_DTYPE)
        eps = pipe.unet(model_input, t, encoder_hidden_states=unconditional).sample.float()
        if guidance > 1:
            # Two sequential calls instead of batch=2.
            eps_cond = pipe.unet(model_input, t, encoder_hidden_states=conditional).sample.float()
            eps = eps + guidance * (eps_cond - eps)
        x = ddim_transfer(x, eps, alpha_from, alpha_to)
    if not torch.isfinite(x).all():
        raise FloatingPointError('DDIM produced NaN/Inf. Restart the runtime and check the model / dtype.')
    result = x.detach().cpu()
    del x, unconditional, conditional
    pipe.maybe_free_model_hooks()
    return result

@torch.inference_mode()
def decode_latent(latent):
    decoded = pipe.vae.decode(
        latent.to(device=DEVICE, dtype=MODEL_DTYPE) / pipe.vae.config.scaling_factor,
        return_dict=False,
    )[0]
    if not torch.isfinite(decoded).all():
        raise FloatingPointError('VAE decode: NaN/Inf. Try an FP32 VAE; see troubleshooting.')
    rgb = ((decoded.float() / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
    pipe.maybe_free_model_hooks()
    return Image.fromarray(rgb, mode='RGB')

@torch.inference_mode()
def encode_image(image):
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    array = np.asarray(image).astype(np.float32) / 127.5 - 1
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(DEVICE, dtype=MODEL_DTYPE)
    # Use mode(), not a random sample(): each image gives a deterministic input state.
    latent = pipe.vae.encode(tensor).latent_dist.mode().float() * pipe.vae.config.scaling_factor
    if not torch.isfinite(latent).all():
        raise FloatingPointError('VAE encode: NaN/Inf.')
    result = latent.detach().cpu()
    del latent, tensor
    pipe.maybe_free_model_hooks()
    return result

def generate_image(noise, prompt):
    latent = diffusion_trajectory(noise, GENERATION_STEPS, prompt=prompt, guidance=GUIDANCE_SCALE)
    return decode_latent(latent)

def recover_noise(image):
    latent = encode_image(image)
    return diffusion_trajectory(latent, DETECTION_STEPS, inverse=True, prompt='', guidance=1.0)

## 6. Detector metrics

Report two values: the mean complex **L1 distance** between the FFT and the key
(lower means closer to the watermark), and the **model-based p-value** from the noncentral
chi-square test in section 3.4 of the paper. `p < ALPHA` indicates a detection under this test.

This is not the probability that an image is fake. Gaussian/i.i.d. assumptions,
dependencies between FFT coefficients, and inversion error limit the interpretation of the p-value.
`ALPHA=0.01` **does not establish an actual FPR of 1%** on these few examples.
Do not tune the threshold after inspecting attacks. A numerical p=0 indicates underflow;
plots use a lower bound of 1e-300.

In [ ]:
def watermark_metrics(recovered, candidate_key, candidate_mask):
    y = fft2c(recovered)[candidate_mask].numpy().astype(np.complex128)
    k = candidate_key[candidate_mask].numpy().astype(np.complex128)
    sigma2 = max(float(np.mean(np.abs(y)**2)), np.finfo(np.float64).tiny)
    eta = float(np.sum(np.abs(k - y)**2) / sigma2)
    noncentrality = float(np.sum(np.abs(k)**2) / sigma2)
    p = float(ncx2.cdf(eta, df=len(y), nc=noncentrality))
    if not math.isfinite(p):
        raise FloatingPointError('Noncentral chi-square returned a nonfinite p-value.')
    return {
        'distance_l1': float(np.mean(np.abs(y - k))),
        'p_paper': p,
        'neg_log10_p': float(-np.log10(max(p, 1e-300))),
        'detected': bool(p < ALPHA),
    }

def detect_image(image):
    start = time.perf_counter()
    recovered = recover_noise(image)
    metrics = watermark_metrics(recovered, key, mask)
    metrics['seconds'] = time.perf_counter() - start
    return metrics, recovered

## 7. Two images from the same initial noise

Both branches use the same prompt and seed. One branch modifies the initial noise,
so scene details may differ. The pixel difference between the images does not measure
watermark imperceptibility and does not replace FID or a user study.

In [ ]:
start = time.perf_counter()
images = {
    'unmarked': generate_image(noise_plain, PROMPT),
    'watermarked': generate_image(noise_marked, PROMPT),
}
for kind, image in images.items():
    image.save(RESULTS / f'{kind}.png')
print(f'Pair generation: {time.perf_counter() - start:.1f} s')
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, (kind, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(kind)
    ax.axis('off')
fig.savefig(RESULTS / '02_generated_pair.png', dpi=150)
plt.show()
plt.close(fig)
memory_rows.append(memory_report('pair_generated'))

In [ ]:
records, clean_recovered = [], {}
for kind, image in images.items():
    metrics, recovered = detect_image(image)
    clean_recovered[kind] = recovered
    records.append({'attack': 'clean', 'kind': kind, **metrics})
display(pd.DataFrame(records))
pd.DataFrame(records).to_csv(RESULTS / 'attack_results.csv', index=False)

# Negative control with a different key: no additional inversion needed.
wrong_key, wrong_mask = make_key(LATENT_SHAPE, KEY_SEED + 12345, W_RADIUS, W_CHANNEL)
key_controls = pd.DataFrame([
    {'key': 'correct', **watermark_metrics(clean_recovered['watermarked'], key, mask)},
    {'key': 'wrong', **watermark_metrics(clean_recovered['watermarked'], wrong_key, wrong_mask)},
])
display(key_controls)
key_controls.to_csv(RESULTS / 'key_controls.csv', index=False)
plot_spectra(
    [noise_marked, clean_recovered['unmarked'], clean_recovered['watermarked']],
    ['Marked initial noise (reference)', 'Recovered: unmarked image', 'Recovered: marked image'],
    '03_recovered_spectra.png',
)
if not records[1]['detected']:
    print('The clean watermark was not detected in this example. Keep this result; do not tune the threshold afterward.')
if records[0]['detected']:
    print('The unmarked image produced a false positive. Account for this when interpreting attacks.')
memory_rows.append(memory_report('clean_detection'))

## 8. Attack the final image

JPEG compression, blur, crop+resize, rotation, and Gaussian noise modify image pixels
**after generation**. Apply the same operation to both branches.
The attack noise is identical for both images; use floating-point arithmetic before clipping.

The distortion families come from the paper's experiments. The specific settings here form
a teaching grid. A centered crop differs from the random crop used by the authors;
a 75° rotation adds black corners, while a 180° rotation preserves the entire frame.
An attack succeeds when the watermarked image loses detection **provided the clean
watermarked image was detected**. Inspect image quality separately.

In [ ]:
ATTACKS = [
    {'name': 'jpeg_q50', 'type': 'jpeg', 'value': 50},
    {'name': 'blur_r2', 'type': 'blur', 'value': 2.0},
    {'name': 'crop_area75', 'type': 'crop', 'value': 0.75},
    {'name': 'noise_std005', 'type': 'noise', 'value': 0.05},
    {'name': 'rotate_75', 'type': 'rotate', 'value': 75},
    {'name': 'rotate_180', 'type': 'rotate', 'value': 180},
]

def apply_attack(image, spec, seed=731):
    kind, value = spec['type'], spec['value']
    image = image.convert('RGB')
    if kind == 'jpeg':
        buffer = io.BytesIO()
        image.save(buffer, format='JPEG', quality=int(value))
        buffer.seek(0)
        with Image.open(buffer) as decoded:
            return decoded.convert('RGB').copy()
    if kind == 'blur':
        return image.filter(ImageFilter.GaussianBlur(radius=float(value)))
    if kind == 'crop':
        if not 0 < value <= 1:
            raise ValueError('Crop value must be the retained area fraction in (0, 1].')
        w, h = image.size
        cw, ch = max(1, round(w * math.sqrt(value))), max(1, round(h * math.sqrt(value)))
        left, top = (w - cw) // 2, (h - ch) // 2
        return image.crop((left, top, left + cw, top + ch)).resize(image.size, Image.Resampling.BICUBIC)
    if kind == 'noise':
        array = np.asarray(image).astype(np.float32) / 255
        noise = np.random.default_rng(seed).normal(0, value, array.shape)
        array = np.rint(np.clip(array + noise, 0, 1) * 255).astype(np.uint8)
        return Image.fromarray(array, mode='RGB')
    if kind == 'rotate':
        return image.rotate(float(value), resample=Image.Resampling.BICUBIC, expand=False, fillcolor=(0, 0, 0))
    if kind == 'brightness':
        return ImageEnhance.Brightness(image).enhance(float(value))
    raise ValueError(f'Unknown attack: {kind}')

attacked_images = {'clean': images.copy()}
for spec in ATTACKS:
    attacked_images[spec['name']] = {kind: apply_attack(image, spec) for kind, image in images.items()}

seconds_per_detection = np.mean([r['seconds'] for r in records])
print(f'Estimated time for the remaining {2 * len(ATTACKS)} inversions: ~{2 * len(ATTACKS) * seconds_per_detection / 60:.1f} min')
# No regeneration. Save an intermediate CSV after each distortion.
for spec in tqdm(ATTACKS, desc='Attacks'):
    for kind, image in attacked_images[spec['name']].items():
        metrics, recovered = detect_image(image)
        records.append({'attack': spec['name'], 'kind': kind, **metrics})
        image.save(RESULTS / f"{spec['name']}_{kind}.png")
        if kind == 'watermarked':
            torch.save(recovered, RESULTS / f"{spec['name']}_recovered_noise.pt")
        del recovered
        pd.DataFrame(records).to_csv(RESULTS / 'attack_results.csv', index=False)
memory_rows.append(memory_report('attacks_complete'))

In [ ]:
results_df = pd.DataFrame(records)
clean_marked_detected = bool(results_df.query("attack == 'clean' and kind == 'watermarked'").iloc[0]['detected'])
summary = results_df.pivot(index='attack', columns='kind', values=['distance_l1', 'p_paper', 'detected'])
display(summary)

attack_verdicts = []
for spec in ATTACKS:
    subset = results_df[results_df['attack'] == spec['name']].set_index('kind')
    survived = bool(subset.loc['watermarked', 'detected'])
    verdict = ('sign retained' if survived else 'detection lost') if clean_marked_detected else 'clean baseline failed: inconclusive'
    attack_verdicts.append({
        'attack': spec['name'], 'verdict': verdict,
        'false_positive_control': bool(subset.loc['unmarked', 'detected']),
    })
display(pd.DataFrame(attack_verdicts))
pd.DataFrame(attack_verdicts).to_csv(RESULTS / 'attack_verdicts.csv', index=False)

order = ['clean'] + [s['name'] for s in ATTACKS]
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
for kind, color, marker in [('unmarked', '#64748b', 'o'), ('watermarked', '#0f766e', 's')]:
    rows = results_df[results_df['kind'] == kind].set_index('attack').loc[order]
    axes[0].plot(order, rows['distance_l1'], marker=marker, color=color, label=kind)
    axes[1].plot(order, rows['neg_log10_p'], marker=marker, color=color, label=kind)
axes[0].set_ylabel('L1 distance (lower = closer to key)')
axes[1].set_ylabel('-log10(p), clipped at 300')
axes[1].axhline(-math.log10(ALPHA), color='#b91c1c', linestyle='--', label=f'p = {ALPHA}')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
    ax.grid(alpha=0.2)
    ax.legend()
fig.savefig(RESULTS / '04_attack_scores.png', dpi=150)
plt.show()
plt.close(fig)

# Show both branches side by side for each distortion.
fig, axes = plt.subplots(len(order), 2, figsize=(8, 3.4 * len(order)), constrained_layout=True)
for row, name in enumerate(order):
    for col, kind in enumerate(['unmarked', 'watermarked']):
        entry = results_df[(results_df['attack'] == name) & (results_df['kind'] == kind)].iloc[0]
        axes[row, col].imshow(attacked_images[name][kind])
        axes[row, col].set_title(f"{name} | {kind}\np={entry.p_paper:.2e}, detected={bool(entry.detected)}")
        axes[row, col].axis('off')
fig.savefig(RESULTS / '05_attack_gallery.png', dpi=120)
plt.show()
plt.close(fig)

## 9. Optional: vary attack strength

Set `RUN_STRENGTH_SWEEP=True` in section 2 and run this cell.
This checks only the already generated watermarked image; these points do not measure FPR.
The plot shows where detection disappears for this particular example.

In [ ]:
sweep_rows = []
if RUN_STRENGTH_SWEEP:
    sweep_grid = {
        'jpeg': [95, 75, 50, 25, 10],
        'crop': [1.0, 0.9, 0.75, 0.5, 0.25],
        'rotate': [0, 15, 45, 75, 90, 180],
    }
    for kind, values in sweep_grid.items():
        for value in tqdm(values, desc=kind):
            spec = {'name': f'{kind}_{value}', 'type': kind, 'value': value}
            attacked = apply_attack(images['watermarked'], spec)
            metrics, recovered = detect_image(attacked)
            sweep_rows.append({'attack_type': kind, 'strength': value, **metrics})
            del recovered
            pd.DataFrame(sweep_rows).to_csv(RESULTS / 'strength_sweep.csv', index=False)
    sweep_df = pd.DataFrame(sweep_rows)
    display(sweep_df)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    for ax, (kind, values) in zip(axes, sweep_grid.items()):
        rows = sweep_df[sweep_df.attack_type == kind]
        ax.plot(rows.strength, rows.neg_log10_p, marker='o')
        ax.axhline(-math.log10(ALPHA), color='red', linestyle='--')
        ax.set(title=kind, xlabel='Quality / retained area / degrees', ylabel='-log10(p)')
        ax.grid(alpha=0.2)
    fig.savefig(RESULTS / '06_strength_sweep.png', dpi=150)
    plt.show()
    plt.close(fig)
else:
    print('Attack strength sweep disabled. The six main attacks have already run.')

## 10. Optional: independent image pairs and ROC/AUC

Set `RUN_MINI_BENCHMARK=True`. Generate new seeds and prompts;
test both branches under clean conditions and JPEG-50.
Construct ROC curves from `-distance_l1`, and calculate detection rates at the unchanged p-value threshold.

This is an exploratory mini-benchmark. Six pairs cannot estimate a 1% FPR,
establish robustness across all images, or support comparison with large-sample AUC results.
Do not select the best key or threshold using these same results. Zero observed
false positives does not imply zero probability of error.

In [ ]:
def roc_points(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)
    positives, negatives = np.sum(labels == 1), np.sum(labels == 0)
    if positives == 0 or negatives == 0:
        raise ValueError('ROC requires both classes.')
    thresholds = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]
    tpr = np.array([np.sum((scores >= t) & (labels == 1)) / positives for t in thresholds])
    fpr = np.array([np.sum((scores >= t) & (labels == 0)) / negatives for t in thresholds])
    ranks = rankdata(scores)
    auc = (ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives)
    return fpr, tpr, float(auc)

benchmark_rows = []
if RUN_MINI_BENCHMARK:
    benchmark_prompts = [
        'A watercolor painting of a lighthouse on a rocky coastline at sunset',
        'A photo of a bowl of oranges on a wooden kitchen table',
        'A mountain lake reflecting snow covered peaks, landscape photograph',
        'A small stone cottage surrounded by a flowering garden, oil painting',
        'A close up photograph of a green tree frog on a leaf',
        'A futuristic train station at night, architectural illustration',
    ]
    jpeg = {'type': 'jpeg', 'value': 50}
    for i in tqdm(range(BENCHMARK_PAIRS), desc='Benchmark pairs'):
        seed = 10000 + i
        prompt = benchmark_prompts[i % len(benchmark_prompts)]
        base = random_noise(seed)
        for kind, initial in [('unmarked', base), ('watermarked', inject_watermark(base, key, mask))]:
            image = generate_image(initial, prompt)
            for attack_name, attacked in [('clean', image), ('jpeg_q50', apply_attack(image, jpeg))]:
                metrics, recovered = detect_image(attacked)
                benchmark_rows.append({'pair': i, 'seed': seed, 'prompt': prompt,
                                       'kind': kind, 'attack': attack_name, **metrics})
                del recovered
                pd.DataFrame(benchmark_rows).to_csv(RESULTS / 'mini_benchmark.csv', index=False)
    bench_df = pd.DataFrame(benchmark_rows)
    display(bench_df.groupby(['attack', 'kind']).agg(
        count=('detected', 'size'), detection_rate=('detected', 'mean'), mean_l1=('distance_l1', 'mean')))
    fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
    for attack_name, rows in bench_df.groupby('attack'):
        fpr, tpr, auc = roc_points((rows.kind == 'watermarked').astype(int), -rows.distance_l1)
        ax.plot(fpr, tpr, marker='.', label=f'{attack_name}: AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='gray')
    ax.set(xlabel='False positive rate', ylabel='True positive rate', title='Exploratory ROC: small sample')
    ax.legend()
    fig.savefig(RESULTS / '07_mini_roc.png', dpi=150)
    plt.show()
    plt.close(fig)
else:
    print('Mini-benchmark disabled. Sections 1–8 and the export below are enough for a first run.')

## 11. Save the experiment

The ZIP contains images, CSV files, plots, settings, library versions, and the recovered
noise for attacks. The teaching key and initial noise are also exported for reproducibility:
this archive is not a secret watermarking service. Model weights are not included.
`memory.csv` reports process RAM at checkpoints and peak CUDA memory tracked by PyTorch;
it is not a measurement of peak system RAM across the entire virtual machine.

In [ ]:
memory_rows.append(memory_report('finished'))
pd.DataFrame(memory_rows).to_csv(RESULTS / 'memory.csv', index=False)
config_export = {
    'run_id_utc': RUN_ID,
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'prompt': PROMPT, 'image_size': IMAGE_SIZE,
    'generation_steps': GENERATION_STEPS, 'detection_steps': DETECTION_STEPS,
    'guidance_scale': GUIDANCE_SCALE, 'image_seed': IMAGE_SEED, 'key_seed': KEY_SEED,
    'watermark_radius': W_RADIUS, 'watermark_channel': W_CHANNEL, 'alpha': ALPHA,
    'memory_mode': MEMORY_MODE, 'attacks': ATTACKS,
    'run_strength_sweep': RUN_STRENGTH_SWEEP,
    'run_mini_benchmark': RUN_MINI_BENCHMARK, 'benchmark_pairs': BENCHMARK_PAIRS,
    'sampler': 'DDIM eta=0, leading, steps_offset=1, set_alpha_to_one=False',
    'state_dtype': 'float32', 'model_dtype': str(MODEL_DTYPE),
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU smoke test',
    'versions': {name: metadata.version(name) for name in PINNED},
}
(RESULTS / 'config.json').write_text(json.dumps(config_export, ensure_ascii=False, indent=2))
torch.save({'key': key, 'mask': mask, 'initial_unmarked': noise_plain, 'initial_marked': noise_marked,
            'recovered_clean': clean_recovered}, RESULTS / 'educational_key_and_latents.pt')
(RESULTS / 'README.txt').write_text(
    'Tree-Ring teaching experiment. See config.json for settings.\n'
    'attack_results.csv: lower distance_l1 means closer to the known key.\n'
    'p_paper is a model-based approximation, not an empirically guaranteed false-positive rate.\n'
    'Clean baseline must be detected before an attack can be called successful.\n'
    'The key is intentionally included for teaching; do not treat it as secret.\n'
    'Paper: https://arxiv.org/abs/2305.20030\n'
    'Authors code: https://github.com/YuxinWenRick/tree-ring-watermark\n'
)
archive = shutil.make_archive(str(RESULTS), 'zip', root_dir=RESULTS)
print('ZIP:', archive, f'({Path(archive).stat().st_size / 2**20:.1f} MiB)')
if DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print('Outside Colab: the archive is available at the path above.')
    else:
        files.download(archive)
else:
    print('Download the ZIP from the Files sidebar in Colab, or set DOWNLOAD_ZIP=True.')

## Interpret the results and choose next steps

- First check the clean pair and the wrong-key control. If the clean baseline fails, the attack result is inconclusive.
- L1 measures proximity to a particular key, not image quality. Rotation and cropping may destroy details.
- A watermark may survive JPEG or blur and weaken after another operation. One image cannot establish robustness.
- Check the unmarked branch too: an image transformation may cause a false positive.
- More rigorous analysis requires independent calibration/test sets and many images.

### Differences from the authors' experiments

Keep the ring key, circular mask, complex embedding, L1 metric, and empty-prompt inversion.
This notebook uses DDIM with eta=0 for generation. The original `run_tree_ring_watermark.py`
uses DPM-Solver for generation, while the paper discusses DDIM.
FP32 state updates, a CPU RNG, modern Diffusers, and CFG evaluation order may change numerical results.
The selected attacks and small sample do not reproduce the full tables, FID, or CLIP Score experiments.

### Troubleshooting

- **No GPU:** connect a GPU runtime; free Colab does not guarantee accelerator availability.
- **CUDA out of memory:** keep `MEMORY_MODE='offload'`, close other GPU workloads,
  restart the runtime, and rerun. Fewer steps reduce execution time but barely change peak VRAM.
- **System RAM exhausted during loading:** use a fresh runtime and do not load a second pipeline.
  In section 3, keep `low_cpu_mem_usage=True`, `variant='fp16'`, `use_safetensors=True`, and the FP16 file filter.
- **401/403 or unavailable weights:** check the mirror page and its terms. Do not switch to Turbo/SDXL
  without adapting the algorithm. For another mirror of the same SD 2.1-base model, update the ID and commit together.
- **ImportError after installation:** restart the runtime and continue with section 2.
- **NaN in VAE:** after loading, run `pipe.vae.to(dtype=torch.float32)` and cast the VAE inputs
  in `encode_image` and `decode_latent` to `pipe.vae.dtype`. Keep the UNet in FP16.
- **Watermark not detected:** check the key, size, channel, 50 steps, clean baseline, and wrong-key control.
  Do not change the threshold after inspecting the attacks to obtain a preferred result.
- **Session interrupted:** intermediate CSV and PNG files remain until the VM is deleted; download them using Files.
  After VM deletion, rerun the notebook. Use Google Drive manually to save a persistent copy.

### Sources

1. [Wen et al., paper](https://arxiv.org/abs/2305.20030).
2. [Authors' implementation and MIT license](https://github.com/YuxinWenRick/tree-ring-watermark).
3. [SD 2.1-base mirror: weights and license](https://huggingface.co/sd2-community/stable-diffusion-2-1-base).
4. [Diffusers 0.35.1: DDIMScheduler](https://huggingface.co/docs/diffusers/v0.35.1/api/schedulers/ddim).
5. [Diffusers: CPU offload and memory](https://huggingface.co/docs/diffusers/v0.35.1/optimization/memory).
6. [Colab: resources and limitations](https://research.google.com/colaboratory/faq.html).

This notebook contains no pregenerated results: images and metrics appear when you run it.

## License for fragments adapted from the official implementation

MIT License

Copyright (c) 2023 Yuxin Wen

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.